In [4]:
import json
import torch.nn as nn
import torch
from model.Transformer import Transformer,encoder_stack,decoder_stack
from model.Transformer_block import encoder_block,decoder_block
from Data.dataset import Dataset

with open("src_vocab.json", "r") as f:
    src_vocab = json.load(f)

with open("tgt_vocab.json", "r") as f:
    tgt_vocab = json.load(f)

embed_dim = 128

encoder_blocks= [encoder_block(embed_dim,1,256,eps=1e-5) for _ in range(3)]
decoder_blocks= [decoder_block(embed_dim,1,256,eps=1e-5) for _ in range(3)]

enc_stack= encoder_stack(encoder_blocks)
dec_stack= decoder_stack(decoder_blocks)

model= Transformer(enc_stack, dec_stack, embed_dim, len(src_vocab), len(tgt_vocab))

device= torch.device("cuda" if torch.cuda.is_available() else "cpu")

model= model.to(device)

In [5]:
from datasets import load_dataset
import re
from torch.utils.data import DataLoader
from Data.dataloader import collate

def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Zà-ÿ\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

ds = load_dataset("Helsinki-NLP/opus-100", "en-it")

data_pair = []

for sample in ds["train"]:
    en= clean(sample["translation"]["en"])
    it= clean(sample["translation"]["it"])

    data_pair.append((en, it))


dataset= Dataset(data_pair,src_vocab_path="src_vocab.json",tgt_vocab_path="tgt_vocab.json", src_vocab=src_vocab,tgt_vocab=tgt_vocab)


train_loader= DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate)



In [ ]:
epochs=10
criterion= torch.nn.CrossEntropyLoss()
optimizer= torch.optim.AdamW(model.parameters(), lr=1e-5)

for epoch in range(epochs):

    running_loss=0
    model.train()

    for batch_idx,batch in enumerate(train_loader):

        optimizer.zero_grad()

        src = batch["src"].to(device)
        tgt_in = batch["tgt_in"].to(device)
        tgt_out = batch["tgt_out"].to(device)

        output= model(src,tgt_in)
        loss= criterion(output.reshape(-1,len(tgt_vocab)),tgt_out.reshape(-1))


        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f}")




